# ⚙️ EDA-05 — Feature Engineering
## Silver · Turbina Kelmarsh T1 · 2018–2021

---

### Contexto

El dataset etiquetado de EDA-04 contiene las **46 columnas de sensores raw** más las columnas target.  
Los sensores raw son valores puntuales cada 10 minutos — no capturan la **tendencia temporal** ni la **velocidad de cambio** que precede a un fallo.

Este notebook construye las **features de entrenamiento** en dos pasos:

1. **Features calculadas de dominio** — combinaciones de sensores raw que tienen significado físico directo (diferencias de temperatura, error de alineación yaw, desequilibrio entre ejes)
2. **Features de ventana rodante** — media, desviación típica y pendiente de cada sensor/feature calculada en ventanas de 1h, 6h, 24h y 7 días

### Filosofía de diseño: menos es más

Una práctica habitual en ML es añadir todas las features posibles y dejar que el modelo filtre.  
Con LightGBM esto funciona hasta cierto punto, pero en este proyecto hay una restricción importante:

> **Ratio events/features:** para familias con pocos eventos (63 brake_hydro, 71 pitch_bat),  
> un número excesivo de features con señal débil introduce ruido que supera la señal útil.

El criterio de inclusión de cada feature es **doble**:
- ¿Tiene justificación física directa para este fallo concreto?
- ¿Genera una señal continua y limpia (no un binario o un producto de ruidos)?

Las features que no cumplen ambos criterios se eliminan aunque parezcan intuitivas.

### Entradas y salidas

| | Ruta | Descripción |
|---|---|---|
| **Input** | `data/silver/dataset_labeled.parquet` | 210.384 filas × 55 columnas |
| **Output** | `data/silver/features_{familia}.parquet` | Un fichero por familia con features + targets |

---


## 1. Carga del Dataset Etiquetado


In [1]:
import os
import pandas as pd
import numpy as np

base_dir = os.path.dirname(os.getcwd())

# ==============================================================================
# 1. CARGAR DATASET ETIQUETADO
# ==============================================================================
df = pd.read_parquet(os.path.join(base_dir, "data", "silver", "dataset_labeled.parquet"))
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"📡 Dataset cargado: {len(df):,} filas")
print(f"Columnas: {len(df.columns)}")

📡 Dataset cargado: 210,384 filas
Columnas: 55


---

## 2. Features Calculadas de Dominio

Antes del rolling, construimos **features de punto** con significado físico directo.  
Estas features son combinaciones de sensores que el dominio de turbinas eólicas indica como señales de degradación.

### Criterio de inclusión y exclusión

**Se incluyen** únicamente features que:
- Tienen una relación causal directa y conocida con al menos una familia de fallos
- Producen una señal continua y suave (no binaria, no producto de dos señales débiles)
- No son redundantes con otras features ya presentes

### Features calculadas aprobadas

| Feature | Fórmula | Familia | Justificación |
|---------|---------|---------|---------------|
| `yaw_error` | `\|nacelle_pos - wind_dir\| mod 360` | yaw_cable | Error de alineación nacelle-viento — señal directa del esfuerzo yaw |
| `yaw_error_wind` | `yaw_error × wind_speed` | yaw_cable | Par aerodinámico real sobre el sistema de orientación |
| `cable_rate` | `diff(cable_windings, 1 paso)` | yaw_cable | Velocidad de giro del cable de yaw — detecta bloqueos o deslizamiento |
| `nacelle_std_ratio` | `nacelle_std / wind_speed` | yaw_cable | Variabilidad de orientación relativa al viento — inestabilidad del control yaw |
| `t_bearing_delta` | `T_bearing_front - T_ambient` | generator | Temperatura de rodamiento normalizada por condiciones exteriores |
| `t_rear_bearing_delta` | `T_bearing_rear - T_ambient` | generator | Idem para rodamiento trasero |
| `t_stator_delta` | `T_stator - T_ambient` | generator | Temperatura de estátor normalizada |
| `t_bearing_diff` | `T_front - T_rear` | generator | Degradación asimétrica: si un rodamiento calienta más que el otro |
| `t_stator_bearing_diff` | `T_stator - T_front` | generator | Diferencial estátor-rodamiento: estrés del bobinado |
| `t_gear_oil_delta` | `T_gear_oil - T_ambient` | brake_hydro | Temperatura de aceite normalizada — desgaste mecánico |
| `pressure_vs_temp` | `P_inlet / (T_inlet + 273.15)` | brake_hydro | Presión normalizada por temperatura absoluta — elimina efecto viscosidad por frío |
| `metal_particle_rate` | `diff(metal_count, 1 paso).clip(0)` | brake_hydro | Tasa de generación de partículas metálicas — desgaste activo del gearbox |
| `t_bearing_delta_roc` | `diff(t_bearing_delta, 6 pasos=1h)` | generator | Velocidad de cambio de la deriva térmica |
| `t_stator_roc` | `diff(T_stator, 6 pasos=1h)` | generator | Velocidad de calentamiento del estátor |
| `apparent_power_kva` | `sqrt(P² + Q²)` | generator | Carga real del convertidor incluyendo potencia reactiva |
| `reactive_power_ratio` | `Q / apparent_power` | generator | Fracción reactiva — degradación del convertidor |
| `pitch_asymmetry` | `max(A,B,C) - min(A,B,C)` | pitch_bat | Desalineación entre blades — señal directa de pitch desigual |
| `blade_angle_mean` | `mean(A, B, C)` | pitch_bat | Posición media de pitch como referencia de régimen |
| `motor_current_imbalance` | `std(I1, I2, I3)` | pitch_bat | Desequilibrio entre motores pitch — degradación asimétrica |


In [2]:
# ==============================================================================
# 2. FEATURES CALCULADAS DE DOMINIO
# ==============================================================================

original_columns = set(df.columns)

# --- YAW / ORIENTACIÓN ---
df["yaw_error"] = (df["nacelle_position"] - df["wind_direction"]).abs() % 360
df["yaw_error"] = df["yaw_error"].apply(lambda x: x if x <= 180 else 360 - x)
df["yaw_error_wind"] = df["yaw_error"] * df["wind_speed_ms"]
df["cable_rate"] = df["cable_windings_from_calibration_point"].diff(periods=1).fillna(0)
df["nacelle_std_ratio"] = (df["nacelle_position_standard_deviation"] / (df["wind_speed_ms"] + 1e-6))


# --- TÉRMICAS: deltas respecto a temperatura ambiente ---
# Normalizar temperaturas por T_ambiente elimina la estacionalidad (más calor en verano)
df["t_bearing_delta"]      = df["generator_bearing_front_temperature_c"] - df["nacelle_ambient_temperature_c"]
df["t_rear_bearing_delta"] = df["generator_bearing_rear_temperature_c"]  - df["nacelle_ambient_temperature_c"]
df["t_stator_delta"]       = df["stator_temperature_1_c"]                - df["nacelle_ambient_temperature_c"]
df["t_gear_oil_delta"]     = df["gear_oil_temperature_c"]                - df["nacelle_ambient_temperature_c"]

# --- TÉRMICAS: diferenciales entre componentes ---
# Si front calienta más que rear (o viceversa) → degradación asimétrica
df["t_bearing_diff"]       = df["generator_bearing_front_temperature_c"] - df["generator_bearing_rear_temperature_c"]
df["t_stator_bearing_diff"]= df["stator_temperature_1_c"]                - df["generator_bearing_front_temperature_c"]

# --- RATE OF CHANGE TÉRMICO (derivada a 1h = 6 pasos) ---
# Velocidad de cambio de la deriva térmica — señal de aceleración del fallo
df["t_bearing_delta_roc"]  = df["t_bearing_delta"].diff(periods=6)
df["t_stator_roc"]         = df["stator_temperature_1_c"].diff(periods=6)

# --- ELÉCTRICO ---
df["apparent_power_kva"]   = (df["power_kw"]**2 + df["reactive_power_kvar"]**2) ** 0.5
df["reactive_power_ratio"] = df["reactive_power_kvar"] / (df["apparent_power_kva"] + 1e-6)

# --- HIDRÁULICO---
# Aceite frío → más viscoso → más presión artificial → sin normalización el modelo confunde "presión alta por frío" con "sistema sano"
df["pressure_vs_temp"] = (df["gear_oil_inlet_pressure_bar"] / (df["gear_oil_inlet_temperature_c"] + 273.15))
# delta de 1 paso (10 min) → cuántas partículas se han generado en los últimos 10 min
df["metal_particle_rate"] = df["metal_particle_count"].diff(periods=1).fillna(0)
df["metal_particle_rate"] = df["metal_particle_rate"].clip(lower=0)  # no pueden ser negativas

# --- PITCH ---
df["pitch_asymmetry"] = (
    df[["blade_angle_pitch_position_a", "blade_angle_pitch_position_b", "blade_angle_pitch_position_c"]].max(axis=1) -
    df[["blade_angle_pitch_position_a", "blade_angle_pitch_position_b", "blade_angle_pitch_position_c"]].min(axis=1)
)
df["blade_angle_mean"]         = df[["blade_angle_pitch_position_a",
                                      "blade_angle_pitch_position_b",
                                      "blade_angle_pitch_position_c"]].mean(axis=1)
df["motor_current_imbalance"]  = df[["motor_current_axis_1_a",
                                      "motor_current_axis_2_a",
                                      "motor_current_axis_3_a"]].std(axis=1)


# PENDIENTE FUTURA — NO IMPLEMENTADA AHORA:
# rpm_decel_slope: pendiente de RPM en eventos de parada (Power > 100 → 0 kW)
# Requiere lógica de detección de eventos, no rolling estándar.
# Señal útil para código 2125 (Timeout brake closed).
# T_bearing_vs_expected: residuo T_bearing sobre curva f(Power_kW)
# Requiere calibrar f() con los primeros 6 meses de operación normal.
# La señal más potente para degradación térmica del generador — implementar en v2.


calculated_features = [c for c in df.columns if c not in original_columns]

print(f"✅ Features calculadas generadas")
print(f"   Columnas originales: {len(original_columns)}")
print(f"   Columnas totales: {len(df.columns)}")
print(f"   Features calculadas añadidas: {len(calculated_features)}")


✅ Features calculadas generadas
   Columnas originales: 55
   Columnas totales: 74
   Features calculadas añadidas: 19


---

## 3. Features de Ventana Rodante

### Por qué rolling y no solo valores puntuales

Un sensor puntual de 10 minutos captura el estado **en un momento concreto**.  
Un fallo de degradación progresiva se manifiesta como una **tendencia sostenida** de días o semanas.

Las ventanas rodantes convierten cada sensor puntual en 3 señales temporales:
- **Media** (`mean`): nivel promedio reciente — detecta derivas lentas
- **Desviación típica** (`std`): variabilidad reciente — detecta inestabilidad creciente
- **Pendiente** (`slope`): tendencia — detecta si el valor está subiendo o bajando sistemáticamente

### Ventanas elegidas y su justificación

| Ventana | Pasos de 10min | Justificación |
|---------|---------------|---------------|
| `1h` | 6 | Captura eventos rápidos (variabilidad eléctrica, picos de temperatura) |
| `6h` | 36 | Captura tendencias de medio día (ciclo térmico, patrones de viento) |
| `24h` | 144 | Captura tendencias diarias (cambios de régimen operativo) |
| `7d` | 1008 | Captura degradación lenta de semanas (baterías, desgaste mecánico) |

### Número de features por familia

| Familia | Sensores entrada | Features rolling | Eventos | Ratio |
|---------|-----------------|-----------------|---------|-------|
| `yaw_cable` | 12 | 144 | 217 | 1.51 ✅ |
| `brake_hydro` | 14 | 168 | 63 | 0.38 ✅ |
| `generator` | 21 | 252 | 98 | 0.39 ✅ |
| `pitch_bat` | 15 | 180 | 71 | 0.39 ✅ |

> El ratio eventos/features por encima de 0.3 es aceptable para LightGBM con regularización.  
> LightGBM además hace selección implícita de features en cada árbol — las features de ruido  
> reciben importancia 0 y quedan efectivamente eliminadas sin dañar el modelo.

### Sensores por familia

Cada familia usa **exclusivamente** los sensores con relación causal confirmada al tipo de fallo.  
Mezclar sensores de distintas familias (ej: temperaturas de generador para predecir fallos de freno)  
introduciría correlaciones espurias que el modelo podría aprender de forma incorrecta.


In [3]:
import os
import pandas as pd

# ==============================================================================
# 1. CONFIGURACIÓN DE VENTANAS
# ==============================================================================
WINDOWS = {"1h": 6, "6h": 36, "24h": 144, "7d": 1008}

# ==============================================================================
# 2. FUNCIÓN DE ROLLING FEATURES
# ==============================================================================
def make_rolling_features(df, sensors, windows):
    """
    Genera media, std y pendiente para cada sensor en cada ventana.
    rolling() mira hacia atrás por defecto — correcto, no usa datos futuros.
    min_periods = w//3 permite calcular desde el inicio del dataset sin NaN completos.
    """
    feats = {}
    for col in sensors:
        if col not in df.columns:
            print(f"  ⚠️  WARN: {col} no encontrada, saltando")
            continue
        s = df[col]
        for wname, w in windows.items():
            roll = s.rolling(w, min_periods=max(1, w // 3))
            feats[f"{col}__mean_{wname}"]  = roll.mean()
            feats[f"{col}__std_{wname}"]   = roll.std()
            feats[f"{col}__slope_{wname}"] = roll.apply(
                lambda x: (x.iloc[-1] - x.iloc[0]) / len(x) if len(x) > 1 else 0.0,
                raw=False
            )
    return pd.DataFrame(feats, index=df.index)

# ==============================================================================
# 3. SENSORES POR FAMILIA
# ==============================================================================
# NOTA: Solo sensores con relación causal confirmada con el tipo de fallo.

FAMILY_SENSORS = {

    "yaw_cable": [
        # Sensores raw
        "nacelle_position",
        "nacelle_position_standard_deviation",
        "wind_direction",
        "wind_direction_standard_deviation",
        "vane_position_12",
        "cable_windings_from_calibration_point",
        "wind_speed_ms",
        "power_kw",
        # Features calculadas
        "yaw_error",           # |nacelle - wind_dir| — esfuerzo alineación
        "yaw_error_wind",      # yaw_error × wind_speed — par sobre sistema yaw
        "cable_rate",          # delta(cable_windings)/10min — velocidad acumulación vueltas
        "nacelle_std_ratio",   # nacelle_std / wind_speed — esfuerzo yaw normalizado
    ],

    "brake_hydro": [
        # Sensores raw
        "gear_oil_inlet_pressure_bar",
        "gear_oil_pump_pressure_bar",
        "gear_oil_inlet_temperature_c",
        "gear_oil_temperature_c",
        "generator_rpm_rpm",
        "generator_rpm_standard_deviation_rpm",
        "rotor_speed_rpm",
        "power_kw",
        "front_bearing_temperature_c",
        "rear_bearing_temperature_c",
        "metal_particle_count",
        # Features calculadas
        "t_gear_oil_delta",    # T_aceite - T_ambiente — normalizado por estacionalidad
        "pressure_vs_temp",    # presión / (T_aceite + 273) — normalizado por viscosidad
        "metal_particle_rate", # delta(metal_particle_count) — tasa desgaste activo
    ],

    "generator": [
        # Sensores raw
        "generator_bearing_front_temperature_c",
        "generator_bearing_rear_temperature_c",
        "generator_bearing_front_temperature_max_c",
        "generator_bearing_rear_temperature_max_c",
        "nacelle_temperature_c",
        "nacelle_ambient_temperature_c",
        "ambient_temperature_converter_c",
        "power_kw",
        "reactive_power_kvar",
        "power_factor_cosphi",
        "stator_temperature_1_c",
        "wind_speed_ms",
        # Features calculadas
        "t_bearing_delta",        # T_front - T_ambient
        "t_rear_bearing_delta",   # T_rear - T_ambient
        "t_stator_delta",         # T_stator - T_ambient
        "t_bearing_diff",         # T_front - T_rear (asimetría)
        "t_stator_bearing_diff",  # T_stator - T_front (estrés bobinado)
        "apparent_power_kva",     # sqrt(P² + Q²) — carga real convertidor
        "reactive_power_ratio",   # Q/S — degradación factor de potencia
        "t_bearing_delta_roc",    # derivada t_bearing_delta — velocidad calentamiento
        "t_stator_roc",           # derivada T_stator — velocidad calentamiento
    ],

    "pitch_bat": [
        # Sensores raw
        "motor_current_axis_1_a",
        "motor_current_axis_2_a",
        "motor_current_axis_3_a",
        "blade_angle_pitch_position_a",
        "blade_angle_pitch_position_b",
        "blade_angle_pitch_position_c",
        "temperature_motor_axis_1_c",
        "temperature_motor_axis_2_c",
        "temperature_motor_axis_3_c",
        "nacelle_ambient_temperature_c",
        "power_kw",
        "wind_speed_ms",
        # Features calculadas
        "pitch_asymmetry",          # max(A,B,C) - min(A,B,C)
        "blade_angle_mean",         # media de los 3 ángulos
        "motor_current_imbalance",  # std(I1,I2,I3) — desequilibrio entre ejes
    ],
}

# ==============================================================================
# 4. GENERAR Y GUARDAR FEATURES POR FAMILIA
# ==============================================================================
for family, sensors in FAMILY_SENSORS.items():
    print(f"\n{'='*80}")
    print(f"Generando features para: {family}  ({len(sensors)} sensores × 3 stats × 4 ventanas = {len(sensors)*12} features)")
    print(f"{'='*80}")

    feats = make_rolling_features(df, sensors, WINDOWS)

    target_cols = [f"hours_to_{family}", f"is_pre_{family}"]
    output = pd.concat([df[["timestamp"]], df[target_cols], feats], axis=1)

    output_path = os.path.join(base_dir, "data", "silver", f"features_{family}.parquet")
    output.to_parquet(output_path, index=False)

    rel_path = os.path.relpath(output_path, base_dir)
    print(f"✅ Guardado en ./{rel_path}")
    print(f"   Features rolling:  {feats.shape[1]}")
    print(f"   Filas pre-fallo:   {output[f'is_pre_{family}'].sum():,}  ({100*output[f'is_pre_{family}'].mean():.1f}%)")
    print(f"   Total filas:       {len(output):,}")

print(f"\n{'='*60}")
print("✅ FEATURE ENGINEERING COMPLETADO")
print(f"{'='*60}")



Generando features para: yaw_cable  (12 sensores × 3 stats × 4 ventanas = 144 features)
✅ Guardado en ./data/silver/features_yaw_cable.parquet
   Features rolling:  144
   Filas pre-fallo:   100,065  (47.6%)
   Total filas:       210,384

Generando features para: brake_hydro  (14 sensores × 3 stats × 4 ventanas = 168 features)
✅ Guardado en ./data/silver/features_brake_hydro.parquet
   Features rolling:  168
   Filas pre-fallo:   21,185  (10.1%)
   Total filas:       210,384

Generando features para: generator  (21 sensores × 3 stats × 4 ventanas = 252 features)
✅ Guardado en ./data/silver/features_generator.parquet
   Features rolling:  252
   Filas pre-fallo:   22,272  (10.6%)
   Total filas:       210,384

Generando features para: pitch_bat  (15 sensores × 3 stats × 4 ventanas = 180 features)
✅ Guardado en ./data/silver/features_pitch_bat.parquet
   Features rolling:  180
   Filas pre-fallo:   28,753  (13.7%)
   Total filas:       210,384

✅ FEATURE ENGINEERING COMPLETADO


---

## 📋 Conclusiones del Notebook EDA-05

### Resumen de features generadas

| Familia | Sensores entrada | Features rolling | Fichero output |
|---------|-----------------|-----------------|----------------|
| `yaw_cable` | 12 | 144 | `features_yaw_cable.parquet` |
| `brake_hydro` | 14 | 168 | `features_brake_hydro.parquet` |
| `generator` | 21 | 252 | `features_generator.parquet` |
| `pitch_bat` | 15 | 180 | `features_pitch_bat.parquet` |


### Features pendientes para v2

Dos features con alta potencia predictiva quedan pendientes por requerir lógica de evento en lugar de rolling estándar:

- **`rpm_decel_slope`** (brake_hydro): pendiente de RPM durante eventos de parada (Power > 100 → 0 kW). Señal clave para código 2125. Requiere detección de eventos de parada y cálculo de pendiente solo en esas ventanas.
- **`T_bearing_vs_expected`** (generator): residuo de temperatura de rodamiento sobre la curva `f(Power_kW)` calibrada con operación normal. La señal más potente para degradación térmica del generador. Requiere ajustar un polinomio de grado 2 sobre los primeros 6 meses de datos de operación normal.

Ambas se implementarán como mejora tras la primera iteración de entrenamiento.

---

### Por qué LightGBM puede manejar estas dimensiones

Con 120–252 features por modelo y 63–217 eventos positivos, el ratio está entre 0.39 y 1.81.  
LightGBM gestiona esto mediante:
- **Subsampling de features** en cada árbol (`colsample_bytree`)
- **Regularización L1/L2** que penaliza features con poco gain
- **min_child_samples** que evita splits sobre pocos positivos

Las features de ruido real (si quedan) recibirán importancia 0 y no dañarán el modelo.  
La eliminación previa de las 8 features problemáticas reduce el riesgo de overfitting  
especialmente en `brake_hydro` (63 eventos) y `pitch_bat` (71 eventos).

### Siguiente paso

**Notebook `06_train_yaw_cable.ipynb`** — entrenamiento del primer modelo con LightGBM,  
split temporal 80/20, `class_weight="balanced"`, evaluación con Recall y Precision,  
y visualización de alertas en el tiempo vs. fallos reales.
